# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

---
## 1. Подготовка

In [3]:
!java -version

[0.012s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.013s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [4]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [5]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Writing DemurrageLegacy.java


In [6]:
!java DemurrageLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [ ]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** _(Замечание о том, что метод calc объявлен static, добавлять не нужно. Статический метод подходит для данной задачи, поскольку расчёт платы за простой не требует создания объекта класса и не использует его нестатические поля.)_

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [7]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Writing ShiftPlanLegacy.java


In [8]:
!java ShiftPlanLegacy.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exception in thread "main" java.util.ConcurrentModificationException
	at java.base/java.util.ArrayList$Itr.checkForComodification(ArrayList.java:1095)
	at java.base/java.util.ArrayList$Itr.next(ArrayList.java:1049)
	at ShiftPlanLegacy.make(ShiftPlanLegacy.java:24)
	at ShiftPlanLegacy.main(ShiftPlanLegacy.java:19)


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [9]:

# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(
        Строка="20-22",
        Категория="Корректность",
        Серьёзность="blocker",
        Замечание="Во время обхода списка brigades через for-each происходит удаление элементов. Это может вызвать ConcurrentModificationException и аварийное завершение формирования наряда.",
        Предложение="Использовать Iterator с remove() или фильтрацию через removeIf()."
    ),
    dict(
        Строка="20-22",
        Категория="Проектирование",
        Серьёзность="major",
        Замечание="Метод make изменяет исходный список brigades, переданный вызывающим кодом. После вызова вызывающая сторона теряет часть данных и не может повторно использовать первоначальный список.",
        Предложение="Создавать отдельный список подходящих бригад, не изменяя исходный."
    ),
    dict(
        Строка="25-27",
        Категория="Корректность",
        Серьёзность="blocker",
        Замечание="Если количество подходящих бригад меньше значения need, обращение brigades.get(i) вызовет IndexOutOfBoundsException и наряд не будет сформирован.",
        Предложение="Проверять количество подходящих бригад до формирования наряда и выбрасывать IllegalArgumentException."
    ),
    dict(
        Строка="30-32",
        Категория="Поведение при ошибках",
        Серьёзность="major",
        Замечание="FileWriter не закрывается после записи. Это может привести к потере данных из-за незаписанного буфера и утечке системного ресурса.",
        Предложение="Использовать try-with-resources для автоматического закрытия файла."
    ),
    dict(
        Строка="33-34",
        Категория="Поведение при ошибках",
        Серьёзность="blocker",
        Замечание="Исключение при записи файла полностью игнорируется пустым catch. Пользователь не узнает, что наряд не был сохранён, хотя метод сообщит об успешном формировании.",
        Предложение="Обрабатывать IOException и передавать понятную ошибку вызывающему коду."
    ),
    dict(
        Строка="9",
        Категория="Корректность",
        Серьёзность="major",
        Замечание="Объект SimpleDateFormat хранится в static-поле и используется совместно. Этот класс не потокобезопасен, поэтому одновременные вызовы make из разных потоков могут привести к некорректному форматированию даты.",
        Предложение="Использовать потокобезопасный DateTimeFormatter из java.time."
    ),
    dict(
        Строка="10, 36",
        Категория="Проектирование",
        Серьёзность="major",
        Замечание="Журнал log является общим изменяемым static-списком. При повторных вызовах make в журнале накапливаются старые записи, а при параллельной работе возможны проблемы с доступом.",
        Предложение="Передавать журнал как отдельный объект или использовать локальное состояние."
    ),
    dict(
        Строка="23-27",
        Категория="Читаемость",
        Серьёзность="minor",
        Замечание="Формирование строки наряда через многократную конкатенацию result = result + ... создаёт лишние промежуточные объекты String и усложняет работу со строкой.",
        Предложение="Использовать StringBuilder и метод append()."
    )
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
ревью


Записано замечаний: 8


,Строка,Категория,Серьёзность,Замечание,Предложение
0,20-22,Корректность,blocker,Во время обхода списка brigades через for-each...,Использовать Iterator с remove() или фильтраци...
1,20-22,Проектирование,major,"Метод make изменяет исходный список brigades, ...","Создавать отдельный список подходящих бригад, ..."
2,25-27,Корректность,blocker,Если количество подходящих бригад меньше значе...,Проверять количество подходящих бригад до форм...
3,30-32,Поведение при ошибках,major,FileWriter не закрывается после записи. Это мо...,Использовать try-with-resources для автоматиче...
4,33-34,Поведение при ошибках,blocker,Исключение при записи файла полностью игнориру...,Обрабатывать IOException и передавать понятную...
5,9,Корректность,major,Объект SimpleDateFormat хранится в static-поле...,Использовать потокобезопасный DateTimeFormatte...
6,"10, 36",Проектирование,major,Журнал log является общим изменяемым static-сп...,Передавать журнал как отдельный объект или исп...
7,23-27,Читаемость,minor,Формирование строки наряда через многократную ...,Использовать StringBuilder и метод append().


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [13]:

%%writefile ShiftPlanFixed.java
import java.io.IOException;
import java.io.StringWriter;
import java.io.Writer;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.time.LocalDate;
import java.time.format.DateTimeFormatter;
import java.util.ArrayList;
import java.util.List;

public class ShiftPlanFixed {

    private static final double MAX_HOURS = 10.0;
    private static final DateTimeFormatter DATE_FORMAT =
            DateTimeFormatter.ofPattern("dd.MM.yyyy");

    private static int failedChecks = 0;

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        try {
            runTests();

            String result = make(brigades, 2);

            // Путь к файлу задаётся в main, а не внутри make.
            Files.writeString(
                    Path.of("наряд.txt"),
                    result,
                    StandardCharsets.UTF_8
            );

            System.out.println("Сформированный наряд:");
            System.out.println(result);
            System.out.println("Журнал: "
                    + DATE_FORMAT.format(LocalDate.now())
                    + " наряд сформирован");

        } catch (IOException e) {
            System.err.println(
                    "Не удалось записать наряд в файл: "
                            + e.getMessage()
            );
        } catch (IllegalArgumentException e) {
            System.err.println("Не удалось сформировать наряд: "
                    + e.getMessage());
        } catch (AssertionError e) {
            System.err.println("Обнаружены ошибки в тестах: "
                    + e.getMessage());
        }
    }

    /**
     * Формирует наряд и возвращает его в виде строки.
     * Метод не выполняет файловых операций.
     */
    public static String make(List<Brigade> brigades, int need) {

        if (brigades == null) {
            throw new IllegalArgumentException(
                    "Список бригад не должен быть null"
            );
        }

        if (need <= 0) {
            throw new IllegalArgumentException(
                    "Количество бригад должно быть положительным"
            );
        }

        // Исходный список не изменяется.
        List<Brigade> suitable = new ArrayList<>();
        List<String> rejected = new ArrayList<>();

        for (Brigade brigade : brigades) {

            if (brigade == null) {
                rejected.add("Неизвестная бригада: запись равна null");
                continue;
            }

            if (!Double.isFinite(brigade.hoursPerShift)
                    || brigade.hoursPerShift < 0) {
                rejected.add(brigade.lastName
                        + " — некорректные данные о количестве часов");
                continue;
            }

            if (!brigade.admitted) {
                rejected.add(brigade.lastName + " — не допущен");
                continue;
            }

            if (brigade.hoursPerShift > MAX_HOURS) {
                rejected.add(brigade.lastName
                        + " — превышены допустимые часы ("
                        + brigade.hoursPerShift + ")");
                continue;
            }

            suitable.add(brigade);
        }

        if (suitable.size() < need) {
            String details = rejected.isEmpty()
                    ? " Причины фильтрации отсутствуют."
                    : " Отфильтрованные бригады: "
                            + String.join("; ", rejected);

            throw new IllegalArgumentException(
                    "Недостаточно подходящих бригад. Доступно: "
                            + suitable.size()
                            + ", требуется: " + need + "."
                            + details
            );
        }

        StringBuilder result = new StringBuilder();

        for (int i = 0; i < need; i++) {
            result.append(suitable.get(i).lastName)
                    .append("; ");
        }

        return result.toString();
    }

    /**
     * Формирует наряд и записывает его в переданный Writer.
     * Вызывающий код сам отвечает за создание и закрытие Writer.
     */
    public static String make(
            List<Brigade> brigades, int need, Writer writer)
            throws IOException {

        if (writer == null) {
            throw new IllegalArgumentException(
                    "Writer не должен быть null"
            );
        }

        String result = make(brigades, need);
        writer.write(result);
        return result;
    }

    // Проверка без немедленного прерывания тестирования.
    public static void check(String name, boolean condition) {
        if (condition) {
            System.out.println("[OK] " + name);
        } else {
            System.out.println("[FAIL] " + name);
            failedChecks++;
        }
    }

    // Проверка ожидаемого исключения.
    public static void checkException(
            String name, Runnable action) {

        boolean exceptionThrown = false;

        try {
            action.run();
        } catch (IllegalArgumentException e) {
            exceptionThrown = true;
        } catch (Exception e) {
            // Неожиданное исключение также означает провал теста.
        }

        check(name, exceptionThrown);
    }

    public static void runTests() {
        failedChecks = 0;

        List<Brigade> testBrigades = new ArrayList<>();
        testBrigades.add(new Brigade("Иванов", 11.0, true));
        testBrigades.add(new Brigade("Петров", 0.0, true));
        testBrigades.add(new Brigade("Сидоров", 4.5, false));
        testBrigades.add(new Brigade("Козлов", 8.0, true));

        // Тест 1. Правильный состав наряда.
        String result = make(testBrigades, 2);
        check("Правильный состав наряда",
                result.equals("Петров; Козлов; "));

        // Тест 2. Исходный список не изменяется.
        check("Исходный список не изменён",
                testBrigades.size() == 4);

        // Тест 3. Недостаточное количество бригад.
        checkException("Проверка недостаточного количества",
                () -> make(testBrigades, 3));

        // Тест 4. Отрицательное количество бригад.
        checkException("Проверка отрицательного количества",
                () -> make(testBrigades, -1));

        // Тест 5. Передача null вместо списка.
        checkException("Проверка null",
                () -> make(null, 1));

        // Тест 6. Отрицательное количество часов.
        List<Brigade> invalidHours = new ArrayList<>();
        invalidHours.add(new Brigade("Тест", -5.0, true));

        checkException("Проверка отрицательных часов",
                () -> make(invalidHours, 1));

        // Тест 7. Проверка NaN.
        List<Brigade> nanHours = new ArrayList<>();
        nanHours.add(new Brigade(
                "Тест NaN", Double.NaN, true));

        checkException("Проверка NaN",
                () -> make(nanHours, 1));

        // Тест 8. Проверка положительной бесконечности.
        List<Brigade> infiniteHours = new ArrayList<>();
        infiniteHours.add(new Brigade(
                "Тест Infinity", Double.POSITIVE_INFINITY, true));

        checkException("Проверка положительной бесконечности",
                () -> make(infiniteHours, 1));

        // Тест 9. Проверка отрицательной бесконечности.
        List<Brigade> negativeInfinityHours = new ArrayList<>();
        negativeInfinityHours.add(new Brigade(
                "Тест -Infinity", Double.NEGATIVE_INFINITY, true));

        checkException("Проверка отрицательной бесконечности",
                () -> make(negativeInfinityHours, 1));

        // Тест 10. Запись через переданный Writer, без реального файла.
        StringWriter testWriter = new StringWriter();
        try {
            String writerResult = make(
                    testBrigades, 2, testWriter);

            check("Проверка записи через Writer",
                    writerResult.equals("Петров; Козлов; ")
                            && testWriter.toString().equals(
                                    "Петров; Козлов; "));
        } catch (IOException e) {
            check("Проверка записи через Writer", false);
        }

        // Итог: сообщаем обо всех проваленных проверках сразу.
        if (failedChecks > 0) {
            throw new AssertionError(
                    "Количество проваленных проверок: "
                            + failedChecks
            );
        }

        System.out.println("Все тесты выполнены успешно.");
    }
}

class Brigade {

    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift,
            boolean admitted) {

        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}


Overwriting ShiftPlanFixed.java


In [14]:
!java ShiftPlanFixed.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[OK] Правильный состав наряда
[OK] Исходный список не изменён
[OK] Проверка недостаточного количества
[OK] Проверка отрицательного количества
[OK] Проверка null
[OK] Проверка отрицательных часов
[OK] Проверка NaN
[OK] Проверка положительной бесконечности
[OK] Проверка отрицательной бесконечности
[OK] Проверка записи через Writer
Все тесты выполнены успешно.
Сформированный наряд:
Петров; Козлов; 
Журнал: 18.11.2026 наряд сформирован


**Таблица прослеживаемости.** Заполните соответствие «замечание → что изменено». Именно она, а не объём правок, показывает качество работы.

|№ замечания| Серьёзность | Что изменено в коде |
|---|---|---|
|1| blocker | Устранено удаление элементов из исходного списка во время обхода. Для отбора бригад создан отдельный список suitable. |
|2| major | Добавлена проверка количества подходящих бригад перед формированием наряда. При недостаточном количестве выбрасывается IllegalArgumentException. |
|3| major | Запись файла переведена на try-with-resources, благодаря чему файл автоматически закрывается после записи. |
|4| blocker | Удалена пустая обработка исключений. Ошибки передачи и записи данных больше не игнорируются. |
|5| major | Вместо SimpleDateFormat используется DateTimeFormatter, а для формирования строк применяется StringBuilder. |

---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [12]:
# 6.1 Ссылки на выполненное ревью (понадобятся самопроверке)
ССЫЛКА_НА_МОЁ_РЕВЬЮ = "https://github.com/polly-wvw/Shust_lab/pull/8#pullrequestreview-5951952443"      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = "https://github.com/Veronikka38/Kuryanovich/pull/9"         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 5       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 6        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 5         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

Ревью: не указано
Свой запрос: не указан


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? |  |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? |  |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? |  |
| Какое из полученных вами замечаний вы отклонили и на каком основании? |  |

### 7.2 Контрольные вопросы

1. Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.
   **Ответ:** _(Ревью — это проверка кода человеком, который анализирует его логику, архитектуру, читаемость и возможные ошибки. Тестирование — это запуск программы с определёнными входными данными для проверки фактического поведения. Статический анализ — автоматическая проверка исходного кода без его выполнения.)_
2. Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?
   **Ответ:** _(Статический анализатор хорошо обнаруживает неиспользуемые переменные, потенциальные утечки ресурсов, неправильную обработку исключений, подозрительные сравнения, возможные ошибки типов и нарушения стандартов оформления кода.Однако он не всегда способен определить ошибки бизнес-логики, неправильность выбранного алгоритма, несоответствие требованиям заказчика и неудобство архитектуры.)_
3. Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.
   **Ответ:** _(Полезное замечание состоит из трёх частей:
   1. Указание конкретного места в коде.
   2. Описание проблемы и её возможных последствий.
   3. Предложение по исправлению.
   Плохая формулировка: «Неправильно работает список».
   Хорошая формулировка: «В методе make при обходе списка brigades происходит удаление его элементов внутри цикла for-each. Это может привести к ConcurrentModificationException и аварийному завершению программы. Рекомендуется использовать отдельный список для подходящих бригад или Iterator с методом remove().»)_
4. Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?
   **Ответ:** _(Approve при наличии незакрытого blocker создаёт ложное ощущение безопасности и подтверждает готовность кода, несмотря на наличие критической ошибки. В результате дефект может попасть в основную ветку проекта и вызвать серьёзные проблемы при эксплуатации. Отсутствие ревью хотя бы не создаёт ложного подтверждения качества кода.)_
5. Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?
   **Ответ:** _(Характеризующий тест фиксирует фактическое поведение существующей программы до внесения изменений. Он необходим для того, чтобы понять, какое поведение нужно сохранить после исправления кода.)_
6. Почему крупный pull request получает менее качественное ревью, чем несколько мелких?
   **Ответ:** _(Крупный pull request содержит много изменений, поэтому рецензенту сложнее внимательно изучить каждую строку и понять взаимосвязи между изменениями. Увеличивается вероятность пропустить ошибку, а также снижается концентрация внимания. Несколько небольших pull request проще проверять, обсуждать и исправлять, поэтому качество ревью обычно выше.)_

### 7.3 Три вывода по работе

1. В ходе лабораторной работы я научилась проводить ревью Java-кода, находить ошибки и оценивать их серьёзность.

2. Научилась формулировать замечания к коду с указанием конкретной строки, последствий проблемы и предложения по исправлению. Также я поняла важность сохранения исходного поведения программы при рефакторинге.

3. В результате работы я получила практические навыки исправления ошибок, написания автоматических проверок, ведения таблицы прослеживаемости и проведения ревью через GitHub. Убедилась, что качественное ревью помогает повысить надёжность и сопровождаемость программного обеспечения.

---
## 8. Чек-лист и самопроверка

- [ ] Образец изучен, отвечен вопрос п. 3.2
- [ ] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [ ] Не менее трёх замечаний серьёзности `blocker` или `major`
- [ ] В каждом замечании указано место, последствие и предложение
- [ ] `ShiftPlanFixed.java` написан, самопроверки проходят
- [ ] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу
- [ ] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request

In [ ]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess

import pandas as pd

ФИО_исполнителя = ""  #@param {type:"string"}
Группа_исполнителя = ""  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = False  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = False  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

строки_отчёта += [
    "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
    "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
    "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
    if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
        ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
if оставлено < 5:
    ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")